In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:22px;}
</style>
"""))

# ※ Quiz : 경주여행과 전주여행에 대해 최빈 단어 시각화와 유사도 분석 시각화

- (1) naver open API를 활용하여 블로그에 "경주여행", "전주여행"을 각각 500건씩 검색하여 백업
    (data/quiz/naver.csv)
     * 백업파일 내용(query, no, title, link, description, total_text(title+ ' ' + description)

- (2) naver.csv에서 total_text를 품사태깅(naver_pos.csv)
     * 파일 내용 : query, no, token, pos

- (3) 명사만 추출(naver_pos_nouns.csv)
     * query, token, pos

- (4) 빈도 분석 백업(naver_pos_nouns_count.csv)
     * token, 경주빈도, 전주빈도, 빈도합     
     
- (5) 빈도 시각화(워드클라우드, Text.plot)
     * 이미지 저장 (data/quiz/)
     
- (6) 단어간 거리 분석(Word2Vec, 연관분석)

## 1. 네이버 open API 활용

In [7]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import pandas as pd
import sys
import urllib.request
from dotenv import load_dotenv
import requests 
import json
load_dotenv() 
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("경주여행")
url = "https://openapi.naver.com/v1/search/blog?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8')[:200])
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Thu, 03 Sep 2026 16:57:24 +0900",
	"total":2735023,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"3월의 <b>경주여행<\/b>.",
			"link":"https:\/\/lje77777.tistory.com\/7132",
			"


In [ ]:
# 문자 -> dict
import json
from html import unescape # description에 있는 &lt;(특수문자)를 <로 변경
import requests
import pandas as pd
import re #특수문자 @@

In [9]:
# 500개 안되서 start1 100 start 2 100
query ='경주 여행'
start = 1

#2번방법 
#url = f"https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}"

In [18]:
#1번방법
url = "https://openapi.naver.com/v1/search/blog.json"
params = {
    'query':query,
    'display':100,
    'start':start
         }
headers = {
    "X-Naver-Client-Id":client_id,
    "X-Naver-Client-Secret":client_secret
          }
response = requests.get(url, headers=headers, params=params)
#items = json.loads(response.text)['items']
items = response.json()['items']
items[:2]

[{'title': '<b>경주여행</b>, 계림 <b>경주</b>역사유적지구 + 황간 대표맛집 유니짜장 덕승관',
  'link': 'https://minui.tistory.com/178',
  'description': '<b>경주여행</b>, 계림<b>경주</b>역사유적지구 + 황간 대표맛집 유니짜장 덕승관 <b>경주 여행</b>에서 빼놓을 수 없는 코스 첨성대가 있는 계림 <b>경주</b>역사유적지구입니다. 한번쯤 수학<b>여행</b>으로도 와본 곳이기도하죠^^ 전 날... ',
  'bloggername': '미루의 공감라이프',
  'bloggerlink': 'https://minui.tistory.com/',
  'postdate': '20190104'},
 {'title': '[<b>경주여행</b>] 교리김밥 봉황대점 - 김밥2줄 9,000원. 쎈데?!ㅋ',
  'link': 'https://jisuni-1116.tistory.com/326',
  'description': '<b>경주여행</b> 먹킷리스트 중에 하나였던 교리김밥 <b>경주</b> 황리단길 끝자락? 무튼 핫플레이스 거리에서 조금만 나가면 교리김밥 봉황대점이있더라. 그 유명한 교리김밥을 먹으로 고고고. 토욜 아점쯤이었는데... ',
  'bloggername': '★먹고 놀자★',
  'bloggerlink': 'https://jisuni-1116.tistory.com/',
  'postdate': '20210429'}]

In [27]:
# title과 description의 <b></b>없애기, html의 특수문자없애기, 일반특수 없애기]
from html import unescape
import re
item =items[4]
title = item.get('title').replace('<b>',' ').replace('</b>',' ')
title = unescape(title)
title = re.sub(r'[^a-zA-Z0-9가-힣]',' ',title)
description=item['description'].replace('<b>',' ').replace('</b>',' ')
description= unescape(description)
description= re.sub(r'[^a-zA-Z0-9가-힣]',' ',description)
link =item['link']
totaltext = title + ' 'description
print(title, description)

  여행  울산  경주 여행  Part4   동궁과 월지  빛누리정원  성호리조트 장생포는 고래문화특구 라는 이름으로 고래를 주제로 한 다채로운 관광지를 형성해두었는데 사실 고래박물관이나 생태체 jinstistory tistory com 2023 08 16     여행  국내 여행       여행  울산  경주 여행  Part3    경주     


In [25]:
# re 정규표현식을 이용해서 특수문자 없애기

title = '[여행] ## & ktx 타고 짱 ㅋㅋ ㅠㅠ'
re.sub(r'[^a-zA-Z0-9가-힣]',' ',title)

' 여행       ktx 타고 짱      '

In [ ]:
# 네이버 API 정보 및 검색 정보
import os
from dotenv import load_dotenv
load_dotenv() 
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
queries = ['경주 여행', '전주 여행']
max_start =5

In [ ]:
# 함수자리!!!
def get_search_element_save(query, start):
    'query와 start로 naver 블로그 검색한 결과로 title link description total_text를 dict list로 반환'

In [ ]:
for query in queries:
    for start in range(1, max_start+1):
    # header와 params와 url 로  request.get(url) -> item   
    # -> title / link / description / total_text
    # ' 검색'  -> 함수 작성